# WAQ .map → NetCDF Conversion & Validation

**Single purpose:** convert each scenario's `.map` binary into a self-contained
NetCDF file with proper (time, M, N, K) dimensions, and verify the conversion
with internal geometric and round-trip sanity checks.

**Inputs:**
- 4 `.map` files (one per scenario)
- `segment number.mat` (segment ID → (M, N, K) lookup)
- `.vol` file (defines ever-active segments)
- `.mdf` parameters (layer thicknesses) — hard-coded from inspection

**Outputs** (one per scenario, saved alongside the `.map` files):
- `<MAP_DIR>/<scen>.map.nc` — (time, M, N, K, substance) with depth coordinate

**Important notes (from the validation iteration that produced this notebook):**

1. **`.map` layout is `(n_seg, n_sub)` per record** — substance index is the
   *last* axis after reshape. The earlier guess of `(n_sub, n_seg)` produced
   garbage values that looked like all substances having identical magnitudes.

2. **`.his` cannot be used for external validation.** The `seg_id` field in
   the `.his` header is *not* a FLOW segment ID; it's an internal monitor
   index. We verified this by observing all CAB monitor seg_ids (3-11) mapping
   to non-water-column grid positions in segnum. Validation here is therefore
   geometric/internal rather than against `.his` ground truth.


## Section 1 — Imports & paths

In [15]:
import os
import struct
import numpy as np
import pandas as pd
import scipy.io as spio
import xarray as xr

# ── Paths ────────────────────────────────────────────────────────────────────
BIN_DIR    = r'<MODEL_DIR>'
MAP_DIR    = os.path.join(BIN_DIR, 'map_files_scenarios')
SEGNUM_MAT = os.path.join(BIN_DIR, 'segment number.mat')
VOL_PATH   = os.path.join(BIN_DIR, 'com_files', 'com-WQexport_30min_fullperiod.vol')

# NetCDFs save alongside the .map files (one .nc per scenario)
NC_OUT_DIR = MAP_DIR
os.makedirs(NC_OUT_DIR, exist_ok=True)

MAP_FILES = {
    'Baseline': 'Torun_WAQ_Apr2023.map',
    'FPV_02':   'Torun_WAQ_Apr2023_scen02.map',
    'FPV_03':   'Torun_WAQ_Apr2023_scen03.map',
    'FPV_04':   'Torun_WAQ_Apr2023_scen04.map',
}

# ── Constants ────────────────────────────────────────────────────────────────
N_SEG_IN = 27904
N_SEG    = 10999
T0       = pd.Timestamp('2022-09-09')
NODATA   = -999.0

# Analysis window — used to subset the validation
ANALYSIS_START = pd.Timestamp('2023-04-01')
ANALYSIS_END   = pd.Timestamp('2023-08-31')

# Active layer range
K_TOP = 7
K_BOT = 15

# Z-model thicknesses from .mdf Thick. FLOW K=0 is bottom; WAQ K=0 is top.
THICK_PCT_FLOW = [12.5, 6.25, 6.25, 6.25, 6.25, 6.25, 6.25, 6.25,
                  3.125, 3.125, 3.125, 3.125, 3.125, 3.125, 12.5, 12.5]
Z_TOTAL = 14.79
THICKNESS_M = {waq_k: THICK_PCT_FLOW[15 - waq_k] / 100.0 * Z_TOTAL
               for waq_k in range(16)}

print(f'.map directory:   {MAP_DIR}')
print(f'NetCDF output:    {NC_OUT_DIR}')
print(f'Analysis window:  {ANALYSIS_START.date()} → {ANALYSIS_END.date()}')

.map directory:   <MODEL_DIR>\map_files_scenarios
NetCDF output:    <MODEL_DIR>\map_files_scenarios
Analysis window:  2023-04-01 → 2023-08-31


## Section 2 — Build `new_to_old` segment mapping

Reads the FLOW `.vol` file to identify which of the 27,904 original segments
are wet at some point during the simulation.

In [16]:
def build_ever_active_mask(vol_path, n_seg_in=N_SEG_IN, n_seg=N_SEG):
    """Scan FLOW .vol; return indices of the first n_seg ever-wet segments."""
    rec_size = (n_seg_in + 1) * 4
    fsize    = os.path.getsize(vol_path)
    n_times  = fsize // rec_size

    ever_active = np.zeros(n_seg_in, dtype=bool)
    print(f'Scanning {n_times:,} timesteps from .vol...')
    with open(vol_path, 'rb') as f:
        for i in range(n_times):
            f.read(4)
            raw = np.frombuffer(f.read(n_seg_in * 4), dtype='<f4').copy()
            ever_active |= (raw > 0)
            if (i + 1) % 2000 == 0:
                print(f'  {i+1:,}/{n_times:,}...', end='\r')

    idx = np.where(ever_active)[0]
    print(f'\n  Total ever-active: {len(idx)}')
    if len(idx) < n_seg:
        raise ValueError(f'Found {len(idx)} ever-active, need {n_seg}')
    return idx[:n_seg]

new_to_old = build_ever_active_mask(VOL_PATH)
print(f'new_to_old: {len(new_to_old)} entries')

Scanning 15,032 timesteps from .vol...
  14,000/15,032...
  Total ever-active: 10999
new_to_old: 10999 entries


## Section 3 — Load segnum + build reconstruction

In [17]:
mat = spio.loadmat(SEGNUM_MAT)
segnum = mat['data'][0, 0]['Val']
M, N, K_FULL = segnum.shape
print(f'segnum shape: ({M}, {N}, {K_FULL})')

valid_mask_3d = ~np.isnan(segnum)
seg_idx_flat  = segnum[valid_mask_3d].astype(np.int64) - 1

flow_to_compact = np.full(N_SEG_IN, -1, dtype=np.int32)
flow_to_compact[new_to_old] = np.arange(N_SEG)

def bin_to_grid_3d(x_bin):
    """Map 1D compact-grid values (length N_SEG) → (M, N, K_FULL) array."""
    x_full = np.full(N_SEG_IN, np.nan, dtype=float)
    x_full[new_to_old] = x_bin
    out = np.full(segnum.shape, np.nan, dtype=float)
    out[valid_mask_3d] = x_full[seg_idx_flat]
    return out

# Per-compact-segment layer index
seg_layer = np.full(N_SEG, -1, dtype=np.int8)
for m in range(M):
    for n in range(N):
        for k in range(K_FULL):
            seg_flow = segnum[m, n, k]
            if np.isnan(seg_flow):
                continue
            compact = flow_to_compact[int(seg_flow) - 1]
            if compact >= 0:
                seg_layer[compact] = k

assert (seg_layer >= 0).all()
print(f'\nCompact-grid segments per layer:')
for k in range(K_FULL):
    n_in = (seg_layer == k).sum()
    tag = '  ← active' if K_TOP <= k <= K_BOT else ''
    print(f'  K={k:2d}: {n_in:5d}{tag}')

segnum shape: (60, 50, 16)

Compact-grid segments per layer:
  K= 0:   104
  K= 1:   161
  K= 2:    45
  K= 3:    37
  K= 4:    25
  K= 5:    36
  K= 6:    22
  K= 7:  1357  ← active
  K= 8:  1326  ← active
  K= 9:  1300  ← active
  K=10:  1246  ← active
  K=11:  1215  ← active
  K=12:  1178  ← active
  K=13:  1105  ← active
  K=14:  1027  ← active
  K=15:   815  ← active


## Section 4 — Read `.map` files (Layout B)

The `.map` binary record layout per timestep is `(n_seg, n_sub)`, not
`(n_sub, n_seg)`. This was confirmed by comparing the two reshape options
against substance-specific value ranges: Layout A produced nearly identical
ranges across all substances (clearly wrong); Layout B produced physically
distinct ranges (OXY in 0-13 mg/L, NH4 in ~0-8 mg/L, etc.) consistent with
real water quality.

**Data shape stored here: `data[time, segment, substance]`**.
Access pattern: `run['data'][ti, :, si]` for "all segments of substance `si`
at timestep `ti`".

In [18]:
def read_map_file(path, n_seg=N_SEG, t0=T0):
    """Read a DELWAQ .map file. Layout is (n_seg, n_sub) per record."""
    with open(path, 'rb') as f:
        titles = [f.read(40).decode('latin-1', errors='replace').strip()
                  for _ in range(4)]
        n_sub, n_seg_file = struct.unpack('<ii', f.read(8))
        if n_seg_file != n_seg:
            raise ValueError(f'.map has {n_seg_file} segments, expected {n_seg}')
        substances = [f.read(20).decode('latin-1', errors='replace').strip()
                      for _ in range(n_sub)]

        header_size = f.tell()
        record_size = 4 + n_sub * n_seg * 4
        fsize = os.path.getsize(path)
        n_times = (fsize - header_size) // record_size
        print(f'  {os.path.basename(path)}: {n_sub} subs × {n_seg:,} segs × {n_times:,} steps')

        # Layout B: each record reshapes as (n_seg, n_sub).
        # We store as (time, n_seg, n_sub) so per-substance access is data[ti, :, si].
        data = np.empty((n_times, n_seg, n_sub), dtype=np.float32)
        timestamps = []
        f.seek(header_size)
        for i in range(n_times):
            ts = struct.unpack('<i', f.read(4))[0]
            data[i] = np.frombuffer(f.read(n_sub * n_seg * 4),
                                    dtype='<f4').reshape(n_seg, n_sub)
            timestamps.append(t0 + pd.Timedelta(seconds=ts))

    return {'substances': substances,
            'idx':        {s: i for i, s in enumerate(substances)},
            'data':       data,
            'timestamps': pd.DatetimeIndex(timestamps),
            'titles':     titles}

print('Loading all .map files...')
map_runs = {}
for name, fname in MAP_FILES.items():
    map_runs[name] = read_map_file(os.path.join(MAP_DIR, fname))

ref = map_runs['Baseline']['substances']
for name, r in map_runs.items():
    assert r['substances'] == ref
print(f'\nSubstances across all runs ({len(ref)}):')
for i, s in enumerate(ref):
    print(f'  [{i:2d}] {s}')

Loading all .map files...
  Torun_WAQ_Apr2023.map: 25 subs × 10,999 segs × 1,253 steps
  Torun_WAQ_Apr2023_scen02.map: 25 subs × 10,999 segs × 1,253 steps
  Torun_WAQ_Apr2023_scen03.map: 25 subs × 10,999 segs × 1,253 steps
  Torun_WAQ_Apr2023_scen04.map: 25 subs × 10,999 segs × 1,253 steps

Substances across all runs (25):
  [ 0] Continuity
  [ 1] OXY
  [ 2] AAP
  [ 3] POC1
  [ 4] PON1
  [ 5] POP1
  [ 6] Opal
  [ 7] NH4
  [ 8] NO3
  [ 9] PO4
  [10] Si
  [11] FDIATOMS_E
  [12] FDIATOMS_P
  [13] FFLAGELA
  [14] GREENS_E
  [15] GREENS_N
  [16] GREENS_P
  [17] MICROCYS_E
  [18] MICROCYS_N
  [19] MICROCYS_P
  [20] AAPS1
  [21] DetCS1
  [22] DetNS1
  [23] DetPS1
  [24] DetSiS1


## Section 5 — Geometric validation (BEFORE converting)

We don't validate against `.his` because the `seg_id` field in the `.his`
header doesn't map back to FLOW segments via segnum (we verified this against
the CAB monitors). Instead, we run internal sanity checks:

1. **CELL COUNT** — total non-NaN cells per timestep matches `new_to_old`
2. **LAYER STRUCTURE** — cells per layer match segnum's expected counts
3. **PHYSICAL RANGES** — each substance falls within plausible bounds
4. **DEPTH GRADIENT** — OXY should be highest near the surface (or at least
   not dramatically lower at the surface than at depth)
5. **CROSS-SCENARIO COHERENCE** — FPV scenarios differ from baseline smoothly

If these all pass, the reconstruction is internally consistent. With the
Layout B fix, substance ranges and the depth gradient should look correct,
unlike under Layout A.

In [19]:
def validate_geometric(map_run, scen, sample_dates=None):
    """Run geometric sanity checks on a single scenario's reconstruction."""
    print(f'\n=== Geometric validation: {scen} ===')

    times = map_run['timestamps']
    in_window = (times >= ANALYSIS_START) & (times <= ANALYSIS_END)
    times_in = times[in_window]
    print(f'  {in_window.sum()} timesteps in window {ANALYSIS_START.date()}'
          f' → {ANALYSIS_END.date()}')

    if sample_dates is None:
        sample_dates = ['2023-04-15', '2023-06-15', '2023-08-15']

    passes = {'cell_count': True, 'layer_counts': True,
              'ranges': True, 'depth_gradient': True}

    # ── Check 1: cell count per timestep ─────────────────────────────────────
    print(f'\n  Check 1: cell count per timestep')
    si_oxy = map_run['idx'].get('OXY')
    for date_str in sample_dates:
        ti = np.argmin(np.abs(map_run['timestamps'] - pd.Timestamp(date_str)))
        x_bin = map_run['data'][ti, :, si_oxy].astype(float)   # Layout B access
        n_valid_bin = ((x_bin > NODATA) & np.isfinite(x_bin)).sum()
        x_bin[x_bin <= NODATA] = np.nan
        grid_3d = bin_to_grid_3d(x_bin)
        n_valid_grid = (~np.isnan(grid_3d)).sum()
        ok = n_valid_grid >= 0.9 * n_valid_bin
        passes['cell_count'] &= ok
        print(f'    {date_str}: bin valid={n_valid_bin:5d}, '
              f'3D grid valid={n_valid_grid:5d}  '
              f'{"✓" if ok else "✗ less than 90% retained"}')

    # ── Check 2: per-layer cell counts ───────────────────────────────────────
    expected_per_layer = {k: int((seg_layer == k).sum()) for k in range(K_FULL)}
    print(f'\n  Check 2: per-layer cell counts (mid-window snapshot)')
    ti_mid = np.where(in_window)[0][len(times_in) // 2]
    x_bin = map_run['data'][ti_mid, :, si_oxy].astype(float)   # Layout B
    x_bin[x_bin <= NODATA] = np.nan
    grid_3d = bin_to_grid_3d(x_bin)
    for k in range(K_FULL):
        n_grid = (~np.isnan(grid_3d[:, :, k])).sum()
        exp = expected_per_layer[k]
        ok = n_grid <= exp
        passes['layer_counts'] &= ok
        if exp > 0:
            print(f'    K={k:2d}: expected ≤{exp:5d}, got {n_grid:5d}  '
                  f'{"✓" if ok else "✗ exceeds segnum"}')

    # ── Check 3: substance range plausibility ────────────────────────────────
    # Bounds are upper limits beyond which values are unphysical — e.g. OXY > 25
    # is non-physical freshwater; NH4 > 50 is sewage-effluent territory.
    bounds = {
        'OXY':      (-1,   25),
        'NH4':      (-0.1, 50),
        'NO3':      (-0.1, 50),
        'PO4':      (-0.01, 10),
        'POC1':     (-1,   500),
        'GREENS_E': (-1,   500),
        'Si':       (-0.1, 30),
    }
    print(f'\n  Check 3: in-bounds fraction per substance')
    for sub, (lo, hi) in bounds.items():
        si = map_run['idx'].get(sub)
        if si is None: continue
        x_bin = map_run['data'][ti_mid, :, si].astype(float)   # Layout B
        valid = (x_bin > NODATA) & np.isfinite(x_bin)
        if valid.sum() == 0: continue
        in_bounds = valid & (x_bin >= lo) & (x_bin <= hi)
        frac = in_bounds.sum() / valid.sum()
        ok = frac > 0.95
        passes['ranges'] &= ok
        v = x_bin[valid]
        print(f'    {sub:10s} [{lo:>6.2f}, {hi:>6.2f}]: '
              f'{frac*100:5.1f}% in bounds  '
              f'(min={v.min():.3g}, max={v.max():.3g}, mean={v.mean():.3g})  '
              f'{"✓" if ok else "✗"}')

    # ── Check 4: depth gradient for OXY ──────────────────────────────────────
    print(f'\n  Check 4: OXY depth gradient (mean per layer)')
    layer_means = {}
    for k in range(K_FULL):
        layer = grid_3d[:, :, k]
        n_in = (~np.isnan(layer)).sum()
        if n_in == 0:
            continue
        layer_means[k] = np.nanmean(layer)
        print(f'    K={k:2d}: mean OXY = {layer_means[k]:.3f}  (n={n_in})')

    active_means = {k: v for k, v in layer_means.items()
                    if K_TOP <= k <= K_BOT}
    if len(active_means) > 1:
        surf = active_means[min(active_means)]
        deep = active_means[max(active_means)]
        ok = surf >= 0.5 * deep    # Allow stratified or anoxic-deep behaviour
        passes['depth_gradient'] &= ok
        print(f'    Surface (K={min(active_means)}) = {surf:.3f}, '
              f'deepest (K={max(active_means)}) = {deep:.3f}  '
              f'{"✓" if ok else "✗ deep > 2× surface (review)"}')

    overall = all(passes.values())
    print(f'\n  → {scen} overall: {"PASS" if overall else "REVIEW"}')
    for k, v in passes.items():
        print(f'    {k}: {"✓" if v else "✗"}')
    return overall


def validate_cross_scenario_consistency(map_runs, substance='OXY'):
    """Check that scenarios differ from baseline smoothly, not chaotically."""
    print(f'\n=== Cross-scenario consistency on {substance} ===')

    ti = np.argmin(np.abs(map_runs['Baseline']['timestamps']
                          - pd.Timestamp('2023-06-15')))
    si = map_runs['Baseline']['idx'][substance]

    base = map_runs['Baseline']['data'][ti, :, si].astype(float)   # Layout B
    base[base <= NODATA] = np.nan

    print(f'  Reference: Baseline {substance} at 2023-06-15')
    for scen in ['FPV_02', 'FPV_03', 'FPV_04']:
        ti_s = np.argmin(np.abs(map_runs[scen]['timestamps']
                                - pd.Timestamp('2023-06-15')))
        v = map_runs[scen]['data'][ti_s, :, si].astype(float)   # Layout B
        v[v <= NODATA] = np.nan
        diff = v - base
        diff_valid = diff[~np.isnan(diff)]
        if diff_valid.size == 0:
            continue
        n_changed = (np.abs(diff_valid) > 0.01).sum()
        pct = n_changed / diff_valid.size * 100
        print(f'    {scen}: mean Δ = {diff_valid.mean():+.4f}, '
              f'max |Δ| = {np.abs(diff_valid).max():.4f}, '
              f'{pct:.1f}% of cells changed by >0.01')


# Run validation
all_pass = True
for scen in MAP_FILES:
    ok = validate_geometric(map_runs[scen], scen)
    all_pass = all_pass and ok

validate_cross_scenario_consistency(map_runs, 'OXY')
validate_cross_scenario_consistency(map_runs, 'NH4')

print('\n' + '='*60)
if all_pass:
    print('Geometric validation PASSED.')
    print('Reconstruction is internally consistent. Proceed to conversion.')
else:
    print('Some checks flagged for review.')
    print('Review before converting — the printout shows which checks failed.')
print('='*60)


=== Geometric validation: Baseline ===
  609 timesteps in window 2023-04-01 → 2023-08-31

  Check 1: cell count per timestep
    2023-04-15: bin valid=10969, 3D grid valid=10969  ✓
    2023-06-15: bin valid=10976, 3D grid valid=10976  ✓
    2023-08-15: bin valid=10973, 3D grid valid=10973  ✓

  Check 2: per-layer cell counts (mid-window snapshot)
    K= 0: expected ≤  104, got   101  ✓
    K= 1: expected ≤  161, got   156  ✓
    K= 2: expected ≤   45, got    42  ✓
    K= 3: expected ≤   37, got    36  ✓
    K= 4: expected ≤   25, got    19  ✓
    K= 5: expected ≤   36, got    34  ✓
    K= 6: expected ≤   22, got    19  ✓
    K= 7: expected ≤ 1357, got  1357  ✓
    K= 8: expected ≤ 1326, got  1326  ✓
    K= 9: expected ≤ 1300, got  1300  ✓
    K=10: expected ≤ 1246, got  1246  ✓
    K=11: expected ≤ 1215, got  1215  ✓
    K=12: expected ≤ 1178, got  1178  ✓
    K=13: expected ≤ 1105, got  1105  ✓
    K=14: expected ≤ 1027, got  1027  ✓
    K=15: expected ≤  815, got   815  ✓

  Check 3

## Section 6 — Convert each scenario to NetCDF

One file per scenario, saved alongside the `.map` files. Contains all
substances on the full (time, M, N, K) grid with NaN for absent cells.

Coords:
- `time` — datetime, matched to `.map` timestamps
- `M`, `N` — integer grid indices (0-based)
- `K` — integer layer (0=top, 15=deepest)
- `depth` — float layer mid-depth in metres (from `.mdf` thicknesses)

Compression: gzip level 4.

In [20]:
# Per-layer mid-depths (computed once, attached as coord)
LAYER_MID_DEPTH_FULL = {}
cum = 0.0
for k in range(K_FULL):
    LAYER_MID_DEPTH_FULL[k] = cum + THICKNESS_M[k] / 2.0
    cum += THICKNESS_M[k]

print('Per-layer mid-depths (full grid, K=0 top):')
for k in range(K_FULL):
    print(f'  K={k:2d}: depth = {LAYER_MID_DEPTH_FULL[k]:6.3f} m  '
          f'(thickness {THICKNESS_M[k]:.3f} m)')

def convert_run_to_netcdf(map_run, scen, out_path):
    """Reconstruct the full (time, M, N, K) array for every substance and
    write to NetCDF. Layout B access: data[ti, :, si]."""
    substances = map_run['substances']
    times = map_run['timestamps']
    n_t = len(times)
    print(f'\n{scen}: building (time={n_t}, M={M}, N={N}, K={K_FULL}) × '
          f'{len(substances)} substances')

    data_vars = {}
    for si, sub in enumerate(substances):
        arr = np.empty((n_t, M, N, K_FULL), dtype=np.float32)
        for ti in range(n_t):
            x_bin = map_run['data'][ti, :, si].astype(float)   # Layout B
            x_bin[x_bin <= NODATA] = np.nan
            arr[ti] = bin_to_grid_3d(x_bin)
            if (ti + 1) % 200 == 0:
                print(f'    {sub}: {ti+1}/{n_t}...', end='\r')
        data_vars[sub] = (['time', 'M', 'N', 'K'], arr,
                          {'long_name': sub, '_FillValue': np.float32(np.nan)})
        print(f'    {sub}: done                                  ')

    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            'time':  times,
            'M':     np.arange(M),
            'N':     np.arange(N),
            'K':     np.arange(K_FULL),
            'depth': ('K', [LAYER_MID_DEPTH_FULL[k] for k in range(K_FULL)]),
        },
        attrs={
            'description':       f'Delft3D-WAQ output, scenario {scen}',
            'source_map':        os.path.basename(map_run.get('source_file', '')),
            'k_convention':      'WAQ K=0 is top (mostly dry), K=15 is deepest',
            'depth_units':       'metres below model Ztop (=water surface at peak)',
            'thickness_source':  'Delft3D-FLOW .mdf Thick (percentages × 14.79 m), reversed for WAQ K',
            'reference_time':    str(T0),
            'binary_layout':     'Source .map records reshaped as (n_seg, n_sub) [Layout B]',
            'created_by':        'WAQ_map_to_netcdf conversion notebook',
        },
    )

    encoding = {v: {'zlib': True, 'complevel': 4} for v in data_vars}
    print(f'  Writing to {out_path}...')
    ds.to_netcdf(out_path, encoding=encoding)
    print(f'  Saved ({os.path.getsize(out_path) / 1e6:.1f} MB)')
    return out_path


# Convert every scenario
written_paths = {}
for scen, fname in MAP_FILES.items():
    map_runs[scen]['source_file'] = fname
    out_path = os.path.join(NC_OUT_DIR, fname + '.nc')
    written_paths[scen] = convert_run_to_netcdf(map_runs[scen], scen, out_path)

print('\nAll NetCDFs written:')
for scen, path in written_paths.items():
    print(f'  {scen:10s}: {path}')

Per-layer mid-depths (full grid, K=0 top):
  K= 0: depth =  0.924 m  (thickness 1.849 m)
  K= 1: depth =  2.773 m  (thickness 1.849 m)
  K= 2: depth =  3.929 m  (thickness 0.462 m)
  K= 3: depth =  4.391 m  (thickness 0.462 m)
  K= 4: depth =  4.853 m  (thickness 0.462 m)
  K= 5: depth =  5.315 m  (thickness 0.462 m)
  K= 6: depth =  5.777 m  (thickness 0.462 m)
  K= 7: depth =  6.240 m  (thickness 0.462 m)
  K= 8: depth =  6.933 m  (thickness 0.924 m)
  K= 9: depth =  7.857 m  (thickness 0.924 m)
  K=10: depth =  8.782 m  (thickness 0.924 m)
  K=11: depth =  9.706 m  (thickness 0.924 m)
  K=12: depth = 10.630 m  (thickness 0.924 m)
  K=13: depth = 11.555 m  (thickness 0.924 m)
  K=14: depth = 12.479 m  (thickness 0.924 m)
  K=15: depth = 13.866 m  (thickness 1.849 m)

Baseline: building (time=1253, M=60, N=50, K=16) × 25 substances
    Continuity: done                                  
    OXY: done                                  
    AAP: done                                  
    

## Section 7 — Post-write round-trip check

Verify that the NetCDFs round-trip cleanly: reopening each file and querying
a sample of (M, N, K) positions returns the same values as the in-memory
reconstruction. This catches any bugs in the xarray conversion, NaN handling,
or compression.

This is a self-consistency check (NetCDF in == NetCDF out == reconstruction),
not an external validation. We've established that external validation
against `.his` isn't possible with the available metadata.

In [21]:
def roundtrip_check(map_run, written_path, scen, n_samples=200, tol_rel=1e-4):
    """For a sample of (ti, m, n, k, si) positions, verify that NetCDF reload
    matches the in-memory reconstruction (which matches the .map at the same
    segment)."""
    print(f'\n=== Round-trip check: {scen} ===')

    with xr.open_dataset(written_path) as ds:
        substances = list(ds.data_vars)
        print(f'  Opened {os.path.basename(written_path)}: '
              f'{len(substances)} substances, {len(ds.time)} timesteps')

        rng = np.random.default_rng(42)
        n_t = len(map_run['timestamps'])
        sample_ti = rng.integers(0, n_t, n_samples)
        sample_sub = rng.choice(substances, n_samples)

        # Use OXY-substance index for filtering, then sample (M, N, K) positions
        # via segnum (just pick wet cells)
        valid_positions = []
        for m in range(M):
            for n in range(N):
                for k in range(K_FULL):
                    if not np.isnan(segnum[m, n, k]):
                        seg_flow = int(segnum[m, n, k]) - 1
                        if flow_to_compact[seg_flow] >= 0:
                            valid_positions.append((m, n, k))
        valid_positions = np.array(valid_positions)
        sample_pos = rng.integers(0, len(valid_positions), n_samples)
        positions = valid_positions[sample_pos]

        failures = []
        for i in range(n_samples):
            ti = int(sample_ti[i])
            sub = sample_sub[i]
            m, n, k = int(positions[i, 0]), int(positions[i, 1]), int(positions[i, 2])

            # NetCDF value
            v_nc = float(ds[sub].isel(time=ti, M=m, N=n, K=k).values)

            # Reconstructed value from in-memory .map
            si = map_run['idx'][sub]
            seg_flow = int(segnum[m, n, k]) - 1
            compact = flow_to_compact[seg_flow]
            v_recon = float(map_run['data'][ti, compact, si])
            if v_recon <= NODATA:
                v_recon = np.nan

            # Compare (allowing both NaN)
            if np.isnan(v_nc) and np.isnan(v_recon):
                continue
            if np.isnan(v_nc) != np.isnan(v_recon):
                failures.append((ti, m, n, k, sub, v_recon, v_nc, 'NaN mismatch'))
                continue

            if abs(v_recon) > 1e-10:
                rel_err = abs(v_recon - v_nc) / abs(v_recon)
            else:
                rel_err = abs(v_recon - v_nc)
            if rel_err > tol_rel:
                failures.append((ti, m, n, k, sub, v_recon, v_nc, f'{rel_err:.2e}'))

    if not failures:
        print(f'  ✓ All {n_samples} sampled positions match exactly')
        return True

    print(f'  ✗ {len(failures)} mismatches out of {n_samples}:')
    for ti, m, n, k, sub, v_r, v_nc, err in failures[:10]:
        print(f'    ti={ti:4d} (M={m:2d}, N={n:2d}, K={k:2d}) {sub:12s}'
              f'  recon={v_r:.4f}  nc={v_nc:.4f}  err={err}')
    if len(failures) > 10:
        print(f'    ... ({len(failures) - 10} more)')
    return False


all_passed = True
for scen, path in written_paths.items():
    ok = roundtrip_check(map_runs[scen], path, scen)
    all_passed = all_passed and ok

print('\n' + '='*60)
if all_passed:
    print('Round-trip check PASSED. NetCDFs are self-consistent.')
else:
    print('Round-trip check FAILED. Check the failures listed above.')
print('='*60)


=== Round-trip check: Baseline ===
  Opened Torun_WAQ_Apr2023.map.nc: 25 substances, 1253 timesteps
  ✓ All 200 sampled positions match exactly

=== Round-trip check: FPV_02 ===
  Opened Torun_WAQ_Apr2023_scen02.map.nc: 25 substances, 1253 timesteps
  ✓ All 200 sampled positions match exactly

=== Round-trip check: FPV_03 ===
  Opened Torun_WAQ_Apr2023_scen03.map.nc: 25 substances, 1253 timesteps
  ✓ All 200 sampled positions match exactly

=== Round-trip check: FPV_04 ===
  Opened Torun_WAQ_Apr2023_scen04.map.nc: 25 substances, 1253 timesteps
  ✓ All 200 sampled positions match exactly

Round-trip check PASSED. NetCDFs are self-consistent.


## Section 8 — Usage examples

Quick reference for using the NetCDFs in downstream analysis.

In [22]:
ds = xr.open_dataset(written_paths['Baseline'])
print('Dataset structure:')
print(ds)

print('\nExample queries:')
print()
print('# Whole-lake mean per layer per time (Hovmöller):')
print("  ds['OXY'].mean(['M', 'N']).plot(x='time', y='depth', yincrease=False)")
print()
print('# Surface snapshot at one date:')
print("  ds['OXY'].sel(K=7).sel(time='2023-07-15', method='nearest').plot()")
print()
print('# Compare scenarios:')
print("  base = xr.open_dataset(written_paths['Baseline'])['OXY']")
print("  fpv2 = xr.open_dataset(written_paths['FPV_02'])['OXY']")
print("  (fpv2 - base).mean(['M', 'N']).plot(x='time', y='depth', yincrease=False)")
print()
print('# Active-layer subset (K=7..15):')
print("  active = ds.sel(K=slice(7, 15))")

ds.close()

Dataset structure:
<xarray.Dataset> Size: 6GB
Dimensions:     (time: 1253, M: 60, N: 50, K: 16)
Coordinates:
  * time        (time) datetime64[ns] 10kB 2023-04-01 ... 2024-02-08
  * M           (M) int64 480B 0 1 2 3 4 5 6 7 8 ... 51 52 53 54 55 56 57 58 59
  * N           (N) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * K           (K) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
    depth       (K) float64 128B ...
Data variables: (12/25)
    Continuity  (time, M, N, K) float32 241MB ...
    OXY         (time, M, N, K) float32 241MB ...
    AAP         (time, M, N, K) float32 241MB ...
    POC1        (time, M, N, K) float32 241MB ...
    PON1        (time, M, N, K) float32 241MB ...
    POP1        (time, M, N, K) float32 241MB ...
    ...          ...
    MICROCYS_P  (time, M, N, K) float32 241MB ...
    AAPS1       (time, M, N, K) float32 241MB ...
    DetCS1      (time, M, N, K) float32 241MB ...
    DetNS1      (time, M, N, K) float32 241MB ...
    DetPS1 